In [12]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 2.4/101.7 MB 11.7 MB/s eta 0:00:09
   - -------------------------------------- 4.2/101.7 MB 9.8 MB/s eta 0:00:10
   -- ------------------------------------- 5.8/101.7 MB 9.4 MB/s eta 0:00:11
   -- ------------------------------------- 7.1/101.7 MB 8.7 MB/s eta 0:00:11
   --- ------------------------------------ 8.1/101.7 MB 7.6 MB/s eta 0:00:13
   --- ------------------------------------ 9.2/101.7 MB 7.3 MB/s eta 0:00:13
   ---- ----------------------------------- 10.2/101.7 MB 7.0 MB/s eta 0:00:13
   ---- ----------------------------------- 11.5/101.7 MB 6.8 MB/s eta 0:00:14
   ----- ---------------------------------- 13.4/101.7 MB 7.1 MB/s eta 0:00:13
   ------ --------------------------------- 15.7/101.7 MB 7.4 MB/s eta 0:00:12
   ------ --------------------------------- 17.6/101.7 MB 7.6 MB/s eta 0:

In [19]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [21]:
train["Title"] = train["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
test["Title"] = test["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)

train["Title"] = train["Title"].replace(
    ["Lady","Countess","Capt","Col","Don","Dr","Major","Rev","Sir","Jonkheer","Dona"], "Rare"
)
test["Title"] = test["Title"].replace(
    ["Lady","Countess","Capt","Col","Don","Dr","Major","Rev","Sir","Jonkheer","Dona"], "Rare"
)

train["Title"] = train["Title"].replace({"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"})
test["Title"] = test["Title"].replace({"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"})

In [22]:
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

train["IsAlone"] = (train["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

In [23]:
train["Age"] = train["Age"].fillna(
    train.groupby(["Sex","Pclass"])["Age"].transform("median")
)
test["Age"] = test["Age"].fillna(
    test.groupby(["Sex","Pclass"])["Age"].transform("median")
)

train["Fare"] = train["Fare"].fillna(train["Fare"].median())
test["Fare"] = test["Fare"].fillna(test["Fare"].median())

train["Embarked"] = train["Embarked"].fillna(train["Embarked"].mode()[0])
test["Embarked"] = test["Embarked"].fillna(test["Embarked"].mode()[0])

In [24]:
train = pd.get_dummies(train, columns=["Sex","Embarked","Title"], drop_first=True)
test = pd.get_dummies(test, columns=["Sex","Embarked","Title"], drop_first=True)

In [25]:
train, test = train.align(test, join='left', axis=1, fill_value=0)

In [26]:
features = [col for col in train.columns if col not in ["Survived","Name","Ticket","Cabin","PassengerId"]]

X = train[features]
y = train["Survived"]
X_test = test[features]

In [27]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X, y)
predictions = model.predict(X_test)

In [28]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)
print("CV Score:", scores.mean())

CV Score: 0.8417550687339151


In [32]:
output = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predictions
})

output.to_csv(" updated submission.csv", index=False)